# Resume NER Training - Google Colab

This notebook trains a BERT-based Named Entity Recognition model for resumes.

## Setup Steps:
1. Enable GPU: Runtime → Change runtime type → GPU
2. Upload your dataset (train.json) to Google Drive
3. Mount Google Drive in Step 2 and update the dataset path
4. Run all cells below

## Step 1: Install Dependencies

In [ ]:
!pip install torch transformers seqeval scikit-learn tqdm numpy

## Step 2: Load Dataset from Google Drive

**Recommended: Use Google Drive** (dataset persists between sessions)

1. Upload `train.json` to your Google Drive
2. Note the file path (e.g., `MyDrive/Resume-NER-Dataset/train.json`)
3. Mount Drive and update the path below

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

# Update the path below to match where your train.json is stored in Drive
# Example: '/content/drive/MyDrive/Resume-NER/data/dataset-5000/train.json'
# Example: '/content/drive/MyDrive/train.json'
DRIVE_DATASET_PATH = '/content/drive/MyDrive/path/to/train.json'  # ← UPDATE THIS PATH

# Copy dataset to local storage (optional - speeds up loading)
# !cp "{DRIVE_DATASET_PATH}" /content/train.json
# print("✅ Dataset copied to local storage")

# Or use directly from Drive (update dataset_path in Step 5)
print(f"\n📁 Dataset location: {DRIVE_DATASET_PATH}")
print("💡 Update DRIVE_DATASET_PATH above to match your file location in Drive")
print("💡 Or uncomment the copy command to copy to local storage for faster access")

## Step 3: Create Required Files

Create the necessary Python files for training

In [ ]:
%%writefile load_new_dataset.py
"""
Converter function to load the new dataset format (train.json) 
and convert it to the format expected by the training pipeline.
"""

import json


def load_new_dataset(json_file_path):
    """
    Load the new dataset format and convert to training format.
    
    New format:
    {
        "text": "...",
        "annotations": [[start, end, "LABEL"], ...]
    }
    
    Expected format (spaCy style):
    [
        (text, {"entities": [(start, end, "LABEL"), ...]}),
        ...
    ]
    """
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    training_data = []
    
    for entry in data:
        if 'text' not in entry or 'annotations' not in entry:
            continue
        
        text = entry['text']
        annotations = entry['annotations']
        
        entities = []
        for ann in annotations:
            if not isinstance(ann, list) or len(ann) != 3:
                continue
            
            start, end, label = ann
            
            # Validate positions
            if start < 0 or end > len(text) or start >= end:
                continue
            
            # Convert to (start, end, label) format expected by training
            entities.append((start, end, label))
        
        if entities:  # Only add entries with valid entities
            training_data.append((text, {"entities": entities}))
    
    return training_data

In [ ]:
%%writefile train_utils.py
"""Training utilities for Resume NER model."""

import re
import torch
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from transformers import BertForTokenClassification, BertTokenizerFast
from torch.optim import Adam
from seqeval.metrics import classification_report
from sklearn.metrics import confusion_matrix
import numpy as np
from tqdm import tqdm, trange


def trim_entity_spans(data):
    """Removes leading and trailing white spaces from entity spans."""
    invalid_span_tokens = re.compile(r'\s')
    cleaned_data = []
    for text, annotations in data:
        entities = annotations['entities']
        valid_entities = []
        for start, end, label in entities:
            valid_start = start
            valid_end = end
            while valid_start < len(text) and invalid_span_tokens.match(text[valid_start]):
                valid_start += 1
            while valid_end > 1 and invalid_span_tokens.match(text[valid_end - 1]):
                valid_end -= 1
            valid_entities.append([valid_start, valid_end, label])
        cleaned_data.append([text, {'entities': valid_entities}])
    return cleaned_data


def get_label(offset, labels):
    if offset[0] == 0 and offset[1] == 0:
        return 'O'
    for label in labels:
        if offset[1] > label[0] and offset[0] < label[1]:
            return label[2]
    return 'O'


def process_resume(data, tokenizer, tag2idx, max_len, is_test=False):
    tok = tokenizer.encode_plus(
        data[0], max_length=max_len, return_offsets_mapping=True, truncation=True)
    curr_sent = {'orig_labels': [], 'labels': []}

    padding_length = max_len - len(tok['input_ids'])

    if not is_test:
        labels = data[1]['entities']
        labels.reverse()
        for off in tok['offset_mapping']:
            label = get_label(off, labels)
            curr_sent['orig_labels'].append(label)
            curr_sent['labels'].append(tag2idx[label])
        curr_sent['labels'] = curr_sent['labels'] + ([0] * padding_length)

    curr_sent['input_ids'] = tok['input_ids'] + ([0] * padding_length)
    curr_sent['token_type_ids'] = tok['token_type_ids'] + ([0] * padding_length)
    curr_sent['attention_mask'] = tok['attention_mask'] + ([0] * padding_length)
    return curr_sent


class ResumeDataset(Dataset):
    def __init__(self, resume, tokenizer, tag2idx, max_len, is_test=False):
        self.resume = resume
        self.tokenizer = tokenizer
        self.is_test = is_test
        self.tag2idx = tag2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.resume)

    def __getitem__(self, idx):
        data = process_resume(
            self.resume[idx], self.tokenizer, self.tag2idx, self.max_len, self.is_test)
        return {
            'input_ids': torch.tensor(data['input_ids'], dtype=torch.long),
            'token_type_ids': torch.tensor(data['token_type_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(data['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(data['labels'], dtype=torch.long),
            'orig_label': data['orig_labels']
        }


def collate_fn(batch):
    """Custom collate function to handle orig_label list field"""
    input_ids = torch.stack([item['input_ids'] for item in batch])
    token_type_ids = torch.stack([item['token_type_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])
    orig_labels = [item['orig_label'] for item in batch]
    
    return {
        'input_ids': input_ids,
        'token_type_ids': token_type_ids,
        'attention_mask': attention_mask,
        'labels': labels,
        'orig_label': orig_labels
    }


def get_hyperparameters(model, ff):
    if ff:
        param_optimizer = list(model.named_parameters())
        no_decay = ["bias", "gamma", "beta"]
        optimizer_grouped_parameters = [
            {
                "params": [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
                "weight_decay_rate": 0.01,
            },
            {
                "params": [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
                "weight_decay_rate": 0.0,
            },
        ]
    else:
        param_optimizer = list(model.classifier.named_parameters())
        optimizer_grouped_parameters = [{"params": [p for n, p in param_optimizer]}]
    return optimizer_grouped_parameters


def get_special_tokens(tokenizer, tag2idx):
    vocab = tokenizer.get_vocab()
    pad_tok = vocab["[PAD]"]
    sep_tok = vocab["[SEP]"]
    cls_tok = vocab["[CLS]"]
    o_lab = tag2idx["O"]
    return pad_tok, sep_tok, cls_tok, o_lab


def flat_accuracy(valid_tags, pred_tags):
    return (np.array(valid_tags) == np.array(pred_tags)).mean()


def train_and_val_model(model, tokenizer, optimizer, epochs, idx2tag, tag2idx, max_grad_norm, device, train_dataloader, valid_dataloader):
    pad_tok, sep_tok, cls_tok, o_lab = get_special_tokens(tokenizer, tag2idx)

    for epoch in range(1, epochs + 1):

        # Training loop with progress bar
        print(f"\n{'='*60}")
        print(f"Epoch {epoch}/{epochs}")
        print(f"{'='*60}")
        print("Starting training loop...")
        model.train()
        tr_loss, tr_accuracy = 0, 0
        nb_tr_examples, nb_tr_steps = 0, 0
        tr_preds, tr_labels = [], []

        # Add tqdm progress bar for batches
        train_progress = tqdm(train_dataloader, desc=f"Training Epoch {epoch}")
        
        for step, batch in enumerate(train_progress):
            b_input_ids, b_input_mask, b_labels = batch['input_ids'], batch['attention_mask'], batch['labels']
            b_input_ids, b_input_mask, b_labels = b_input_ids.to(device), b_input_mask.to(device), b_labels.to(device)

            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
            loss, tr_logits = outputs[:2]

            loss.backward()
            tr_loss += loss.item()
            nb_tr_examples += b_input_ids.size(0)
            nb_tr_steps += 1

            preds_mask = ((b_input_ids != cls_tok) & (b_input_ids != pad_tok) & (b_input_ids != sep_tok))

            tr_logits = tr_logits.cpu().detach().numpy()
            tr_label_ids = torch.masked_select(b_labels, (preds_mask == 1))
            preds_mask = preds_mask.cpu().detach().numpy()
            tr_batch_preds = np.argmax(tr_logits[preds_mask.squeeze()], axis=1)
            tr_batch_labels = tr_label_ids.to("cpu").numpy()
            tr_preds.extend(tr_batch_preds)
            tr_labels.extend(tr_batch_labels)

            tmp_tr_accuracy = flat_accuracy(tr_batch_labels, tr_batch_preds)
            tr_accuracy += tmp_tr_accuracy

            # Update progress bar with loss and accuracy
            train_progress.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{tmp_tr_accuracy:.4f}'
            })

            torch.nn.utils.clip_grad_norm_(parameters=model.parameters(), max_norm=max_grad_norm)
            optimizer.step()
            model.zero_grad()

        tr_loss = tr_loss / nb_tr_steps
        tr_accuracy = tr_accuracy / nb_tr_steps
        print(f"\n✅ Train loss: {tr_loss:.4f}")
        print(f"✅ Train accuracy: {tr_accuracy:.4f}")

        # Validation loop with progress bar
        print("\nStarting validation loop...")
        model.eval()
        eval_loss, eval_accuracy = 0, 0
        nb_eval_steps, nb_eval_examples = 0, 0
        predictions, true_labels = [], []
        pred_tags_sequences, valid_tags_sequences = [], []

        val_progress = tqdm(valid_dataloader, desc=f"Validation Epoch {epoch}")
        
        for batch in val_progress:
            b_input_ids, b_input_mask, b_labels = batch['input_ids'], batch['attention_mask'], batch['labels']
            b_input_ids, b_input_mask, b_labels = b_input_ids.to(device), b_input_mask.to(device), b_labels.to(device)

            with torch.no_grad():
                outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask, labels=b_labels)
                tmp_eval_loss, logits = outputs[:2]

            batch_size = b_input_ids.size(0)
            logits = logits.cpu().detach().numpy()
            b_labels_np = b_labels.cpu().numpy()
            b_input_ids_np = b_input_ids.cpu().numpy()
            
            batch_preds = []
            batch_labels = []
            
            for i in range(batch_size):
                sample_mask = ((b_input_ids_np[i] != cls_tok) & (b_input_ids_np[i] != pad_tok) & (b_input_ids_np[i] != sep_tok))
                sample_preds = np.argmax(logits[i][sample_mask], axis=1)
                sample_labels = b_labels_np[i][sample_mask]
                
                sample_pred_tags = [idx2tag[pred] for pred in sample_preds]
                sample_valid_tags = [idx2tag[label] for label in sample_labels]
            
                pred_tags_sequences.append(sample_pred_tags)
                valid_tags_sequences.append(sample_valid_tags)
                
                batch_preds.extend(sample_preds)
                batch_labels.extend(sample_labels)
                predictions.extend(sample_preds)
                true_labels.extend(sample_labels)

            tmp_eval_accuracy = flat_accuracy(batch_labels, batch_preds)
            eval_loss += tmp_eval_loss.mean().item()
            eval_accuracy += tmp_eval_accuracy

            # Update progress bar
            val_progress.set_postfix({
                'loss': f'{tmp_eval_loss.mean().item():.4f}',
                'acc': f'{tmp_eval_accuracy:.4f}'
            })

            nb_eval_examples += b_input_ids.size(0)
            nb_eval_steps += 1

        pred_tags = [idx2tag[i] for i in predictions]
        valid_tags = [idx2tag[i] for i in true_labels]
        cl_report = classification_report(valid_tags_sequences, pred_tags_sequences)
        eval_loss = eval_loss / nb_eval_steps
        eval_accuracy = eval_accuracy / nb_eval_steps

        print(f"\n✅ Validation loss: {eval_loss:.4f}")
        print(f"✅ Validation Accuracy: {eval_accuracy:.4f}")
        print(f"\nClassification Report:\n{cl_report}")

## Step 4: Download BERT Tokenizer Vocab

We need the vocab.txt file for the tokenizer

In [ ]:
!wget https://huggingface.co/bert-base-uncased/resolve/main/vocab.txt -O vocab.txt
print("✅ vocab.txt downloaded successfully!")

## Step 5: Train the Model

In [ ]:
import torch
from transformers import BertForTokenClassification, BertTokenizerFast
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler
from torch.optim import Adam
from load_new_dataset import load_new_dataset
from train_utils import trim_entity_spans, ResumeDataset, get_hyperparameters, train_and_val_model, collate_fn

# Configuration
MAX_LEN = 500
EPOCHS = 5  # Change this if needed
MAX_GRAD_NORM = 1.0
MODEL_NAME = 'bert-base-uncased'
BATCH_SIZE = 8
VAL_BATCH_SIZE = 4
TRAIN_SPLIT = 0.9

# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("="*60)
print("SYSTEM STATUS")
print("="*60)
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Warning: Using CPU - training will be VERY slow!")
    print("Recommendation: Runtime → Change runtime type → GPU")
print("="*60)

# Setup tokenizer
tokenizer = BertTokenizerFast('vocab.txt', lowercase=True)

# Define labels
NEW_LABELS = [
    "SKILL", "DESIGNATION", "LOCATION", "EXPERIENCE", "PERSON", 
    "EDUCATION", "EXPERTISE", "EMAIL", "COMPANY", "COLLABORATION",
    "LANGUAGE", "ACTION", "CERTIFICATION", "OTHER"
]

tags_vals = ["UNKNOWN", "O"] + NEW_LABELS
tag2idx = {t: i for i, t in enumerate(tags_vals)}
idx2tag = {i: t for i, t in enumerate(tags_vals)}

print(f"\nNumber of labels: {len(tag2idx)}")
print(f"Labels: {NEW_LABELS}")

In [ ]:
# Load dataset
print("Loading dataset...")

# Option 1: Load from Google Drive (if mounted in Step 2)
# Update this path to match your Google Drive location
dataset_path = '/content/drive/MyDrive/path/to/train.json'  # ← UPDATE THIS PATH

# Option 2: Load from local (if you copied it in Step 2)
# dataset_path = '/content/train.json'

# Option 3: If uploaded via Colab UI, use:
# dataset_path = '/content/train.json'

print(f"Dataset path: {dataset_path}")
data = load_new_dataset(dataset_path)
print(f"Loaded {len(data)} entries")

# Clean entity spans
data = trim_entity_spans(data)
print(f"After cleaning: {len(data)} entries")

# Split into train/val
total = len(data)
split_idx = int(total * TRAIN_SPLIT)
train_data, val_data = data[:split_idx], data[split_idx:]

print(f"\nTrain: {len(train_data)} entries")
print(f"Validation: {len(val_data)} entries")

In [ ]:
# Create datasets
train_d = ResumeDataset(train_data, tokenizer, tag2idx, MAX_LEN)
val_d = ResumeDataset(val_data, tokenizer, tag2idx, MAX_LEN)

train_sampler = RandomSampler(train_d)
train_dl = DataLoader(train_d, sampler=train_sampler, batch_size=BATCH_SIZE, collate_fn=collate_fn)
val_dl = DataLoader(val_d, batch_size=VAL_BATCH_SIZE, collate_fn=collate_fn)

print("✅ Datasets created successfully!")
print(f"\nTraining batches: {len(train_dl)}")
print(f"Validation batches: {len(val_dl)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"\nExpected training times:")
if torch.cuda.is_available():
    print(f"  With GPU: ~10-15 minutes per epoch")
else:
    print(f"  With CPU: ~1-2 hours per epoch")
print(f"  Total ({EPOCHS} epochs): ~{EPOCHS * (15 if torch.cuda.is_available() else 90)} minutes")

In [ ]:
# Initialize model
print("Initializing model...")
model = BertForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(tag2idx))
model.to(device)

optimizer_grouped_parameters = get_hyperparameters(model, True)
optimizer = Adam(optimizer_grouped_parameters, lr=3e-5)

print("✅ Model initialized!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Reload train_utils to ensure latest version is used (fixes import issues)
import importlib
import train_utils
importlib.reload(train_utils)
# Re-import the function from the reloaded module
from train_utils import train_and_val_model

# Train model
print("Starting training...")
train_and_val_model(
    model,
    tokenizer,
    optimizer,
    EPOCHS,
    idx2tag,
    tag2idx,
    MAX_GRAD_NORM,
    device,
    train_dl,
    val_dl
)

In [ ]:
# Save model
print("\nSaving model...")
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "tag2idx": tag2idx,
        "idx2tag": idx2tag,
        "model_name": MODEL_NAME
    },
    '/content/model-state.bin'
)
print("✅ Model saved to /content/model-state.bin")

# Download model (optional - to save to your computer)
# from google.colab import files
# files.download('/content/model-state.bin')

## Step 7: Save Model

After training completes, download the model to your computer

In [ ]:
# Uncomment to download model
# from google.colab import files
# files.download('/content/model-state.bin')